# Sesión 6 · Usar APIs de IA

**Módulo 3 · Modelos y APIs** · 2 sesiones · LLMs · prompt engineering · integración

> Hoy programamos de verdad con modelos de lenguaje. Verás el mismo ejemplo con **tres proveedores** para que entiendas que el concepto es idéntico y solo cambian los detalles:
> 1. **Claude (Anthropic)** — de pago, con saldo de prueba gratuito al registrarte
> 2. **OpenRouter** — en la nube, con modelos **gratuitos** (Llama y otros)
> 3. **Ollama** — Llama ejecutándose **en tu propio equipo**, sin clave ni internet

**Al terminar la sesión sabrás:**
- Llamar a una API de IA desde Python (en sus tres variantes)
- Ajustar los parámetros clave: temperatura, tokens y contexto
- Manejar errores con elegancia
- Resolver tres casos reales: chatbot, resumen y clasificación

*Resultado de la sesión: llamas a una API de IA desde Python y construyes tu primer chatbot.*

## ⚠️ Antes de empezar: una nota sincera sobre este notebook

Las celdas que llaman a las APIs **necesitan tus propias claves** (o, para Ollama, el modelo instalado en tu máquina), así que **no se ejecutan solas**: tendrás que poner tu clave cuando se te pida. El código está escrito siguiendo la documentación oficial vigente de cada proveedor (junio 2026), pero recuerda que **estos servicios cambian**: si un nombre de modelo deja de existir, búscalo en la web del proveedor y sustitúyelo. Te indico en cada caso dónde mirar.

**Regla de oro de seguridad:** nunca escribas tu clave directamente en el código ni la subas a internet. Usaremos `getpass` para introducirla sin que quede guardada en el notebook.

## 1. Instalación de las librerías

Ejecuta esta celda una vez. Instala los tres clientes. En Google Colab funciona tal cual.

In [ ]:
# El signo ! ejecuta un comando de terminal desde el notebook
!pip install anthropic openai ollama -q
print("Librerías instaladas ✅")

## 2. Configurar las claves de forma segura

`getpass` pide la clave por teclado sin mostrarla en pantalla. Solo tienes que rellenar las de los servicios que vayas a usar.

- **Anthropic** — saca tu clave en `console.anthropic.com` (empieza por `sk-ant-...`)
- **OpenRouter** — saca tu clave en `openrouter.ai/keys` (empieza por `sk-or-...`)
- **Ollama** — no necesita clave (corre en tu equipo)

In [ ]:
import os
from getpass import getpass

# Introduce solo las que vayas a usar; pulsa Enter para saltar una
os.environ["ANTHROPIC_API_KEY"] = getpass("Clave de Anthropic (Enter para saltar): ")
os.environ["OPENROUTER_API_KEY"] = getpass("Clave de OpenRouter (Enter para saltar): ")
print("Claves configuradas (las que hayas introducido).")

## 3. La misma llamada con los tres proveedores

Vamos a hacer la pregunta más sencilla posible a cada uno. Observa que **la estructura es siempre la misma**: un mensaje del usuario, el modelo responde, leemos el texto.

### Variante A · Claude (Anthropic)

In [ ]:
import anthropic

cliente_claude = anthropic.Anthropic()   # lee ANTHROPIC_API_KEY del entorno

respuesta = cliente_claude.messages.create(
    model="claude-haiku-4-5",            # modelo rápido y económico (puedes usar claude-sonnet-4-6)
    max_tokens=300,
    messages=[
        {"role": "user", "content": "Explícame qué es una API como si tuviera 12 años."}
    ],
)

print(respuesta.content[0].text)

### Variante B · OpenRouter (modelo gratuito en la nube)

OpenRouter ofrece una API **compatible con la de OpenAI**, así que usamos la librería `openai` cambiando la dirección base. Los modelos gratuitos terminan en `:free`.

> ⚠️ La lista de modelos gratuitos cambia con frecuencia. Si el de abajo diera error, entra en `openrouter.ai/models`, filtra por *free* y copia uno que termine en `:free`. Los modelos gratuitos tienen límites de uso (peticiones por minuto y por día).

In [ ]:
from openai import OpenAI

cliente_openrouter = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

respuesta = cliente_openrouter.chat.completions.create(
    model="meta-llama/llama-4-maverick:free",   # modelo Llama gratuito
    messages=[
        {"role": "user", "content": "Explícame qué es una API como si tuviera 12 años."}
    ],
)

print(respuesta.choices[0].message.content)

### Variante C · Ollama (Llama 100% local)

Si has instalado Ollama (`ollama.com`) y descargado un modelo con `ollama pull llama3.2`, puedes ejecutar Llama **en tu propio ordenador, sin internet ni clave**. Ideal para datos sensibles.

> Requiere tener el servicio de Ollama en marcha. Modelos ligeros: `llama3.2`. Más grande y capaz: `llama3.3` (necesita más memoria).

In [ ]:
from ollama import chat

respuesta = chat(
    model="llama3.2",   # debes haberlo descargado antes con: ollama pull llama3.2
    messages=[
        {"role": "user", "content": "Explícame qué es una API como si tuviera 12 años."}
    ],
)

print(respuesta.message.content)

### 💡 Lo importante de los tres bloques

Fíjate en el patrón común: **un cliente, una lista de mensajes con `role` y `content`, y leer el texto de la respuesta**. Lo único que cambia entre proveedores es el nombre del método y dónde está el texto de salida:

| | Crear cliente | Llamar | Leer texto |
|---|---|---|---|
| Claude | `anthropic.Anthropic()` | `cliente.messages.create(...)` | `r.content[0].text` |
| OpenRouter | `OpenAI(base_url=..., api_key=...)` | `cliente.chat.completions.create(...)` | `r.choices[0].message.content` |
| Ollama | `from ollama import chat` | `chat(...)` | `r.message.content` |

## 4. Una función única para no repetir código

Para el resto de la sesión, definimos **una sola función** `preguntar()` que sirve para los tres proveedores. Así nos centramos en los conceptos y no en la fontanería. Apóyate en ella en los ejercicios.

In [ ]:
def preguntar(prompt, proveedor="claude", system=None, temperatura=0.7, max_tokens=500):
    """Envía un prompt al proveedor elegido y devuelve el texto de la respuesta.

    proveedor: 'claude', 'openrouter' u 'ollama'
    system:    instrucciones de rol (opcional)
    """
    if proveedor == "claude":
        cliente = anthropic.Anthropic()
        kwargs = {"model": "claude-haiku-4-5", "max_tokens": max_tokens,
                  "temperature": temperatura,
                  "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        return cliente.messages.create(**kwargs).content[0].text

    elif proveedor == "openrouter":
        cliente = OpenAI(base_url="https://openrouter.ai/api/v1",
                         api_key=os.environ.get("OPENROUTER_API_KEY"))
        mensajes = []
        if system:
            mensajes.append({"role": "system", "content": system})
        mensajes.append({"role": "user", "content": prompt})
        r = cliente.chat.completions.create(model="meta-llama/llama-4-maverick:free",
                                            messages=mensajes, temperature=temperatura,
                                            max_tokens=max_tokens)
        return r.choices[0].message.content

    elif proveedor == "ollama":
        mensajes = []
        if system:
            mensajes.append({"role": "system", "content": system})
        mensajes.append({"role": "user", "content": prompt})
        return chat(model="llama3.2", messages=mensajes).message.content

    else:
        raise ValueError("proveedor debe ser 'claude', 'openrouter' u 'ollama'")

# Prueba (cambia el proveedor según la clave que tengas)
print(preguntar("Dame 3 ideas de negocio sostenible en una frase cada una.", proveedor="claude"))

## 5. Los parámetros que de verdad importan

**Temperatura** — controla la creatividad. Va de 0 a 1 (aprox.):
- Cerca de **0** → respuestas más deterministas y centradas. Úsala para clasificar, extraer datos o cuando quieras precisión
- Cerca de **0.8-1** → respuestas más variadas y creativas. Úsala para lluvia de ideas o redacción

**max_tokens** — el largo máximo de la respuesta. Recuerda: un token ≈ 3 o 4 letras. Limitarlo controla el coste y evita respuestas eternas.

**Contexto / historial** — lo que el modelo "ve". Si quieres que recuerde la conversación, debes **acumular los mensajes** y enviarlos todos. Lo vemos en el chatbot.

In [ ]:
# Mismo prompt, dos temperaturas. Ejecútalo un par de veces y compara.
creativo = preguntar("Inventa un nombre para una app de huertos urbanos.",
                     proveedor="claude", temperatura=1.0)
preciso  = preguntar("Inventa un nombre para una app de huertos urbanos.",
                     proveedor="claude", temperatura=0.0)
print("Temperatura alta :", creativo)
print("Temperatura baja :", preciso)

## 6. Manejar errores con elegancia

Las llamadas a APIs **fallan a veces**: se cae la red, se agota el límite gratuito (error 429), la clave es incorrecta... Un programa serio no se rompe: **captura el error y reacciona**. Para eso usamos `try` / `except`.

In [ ]:
def preguntar_seguro(prompt, proveedor="claude"):
    try:
        return preguntar(prompt, proveedor=proveedor)
    except Exception as error:
        # En un caso real podrías reintentar, esperar, o probar otro proveedor
        return f"⚠️ No se pudo completar la llamada ({type(error).__name__}): {error}"

print(preguntar_seguro("Hola, ¿funcionas?", proveedor="claude"))

## 7. Caso 1 · Chatbot con memoria

Un chatbot necesita **recordar** lo dicho. El truco, como adelantamos: guardar todos los mensajes en una lista e **ir añadiéndolos**. Cada turno enviamos el historial completo.

Aquí lo hacemos con Claude. Para OpenRouter/Ollama la idea es idéntica (lista de mensajes con `role` y `content`).

In [ ]:
historial = [
    {"role": "user", "content": "Hola, me llamo Marta y monto un vivero de plantas."},
]

cliente = anthropic.Anthropic()
r1 = cliente.messages.create(model="claude-haiku-4-5", max_tokens=200, messages=historial)
respuesta1 = r1.content[0].text
print("🤖", respuesta1)

# Guardamos la respuesta del asistente y añadimos un nuevo turno del usuario
historial.append({"role": "assistant", "content": respuesta1})
historial.append({"role": "user", "content": "¿Recuerdas cómo me llamo y a qué me dedico?"})

r2 = cliente.messages.create(model="claude-haiku-4-5", max_tokens=200, messages=historial)
print("🤖", r2.content[0].text)   # debería recordar a Marta y su vivero

## 8. Caso 2 · Resumen automático

Un uso clásico: pegar un texto largo y pedir un resumen. Aquí brilla un **system prompt** que fije el estilo del resumen.

In [ ]:
texto = """La transformación digital no consiste solo en comprar tecnología, sino en
repensar procesos, cultura y modelo de negocio. Muchas pymes empiezan digitalizando tareas
administrativas y, poco a poco, incorporan datos para tomar mejores decisiones. El error
habitual es centrarse en la herramienta y olvidar a las personas que deben usarla."""

resumen = preguntar(
    prompt=f"Resume el siguiente texto en una sola frase clara:\n\n{texto}",
    proveedor="claude",
    system="Eres un experto en sintetizar ideas. Respondes en español, breve y directo.",
    temperatura=0.2,   # baja: queremos fidelidad, no creatividad
)
print(resumen)

## 9. Caso 3 · Clasificación con few-shot

Recuperamos el *few-shot* de la Sesión 5, ahora en código. Pedimos que devuelva **solo la categoría** para poder usar el resultado en un programa.

In [ ]:
def clasificar_sentimiento(comentario, proveedor="claude"):
    prompt = f"""Clasifica el sentimiento como POSITIVO, NEGATIVO o NEUTRO.
Responde SOLO con una de esas tres palabras, sin explicaciones.

Comentario: "El producto llegó tarde y mal embalado"  -> NEGATIVO
Comentario: "Funciona perfectamente, muy contento"     -> POSITIVO
Comentario: "Es lo que esperaba, sin más"              -> NEUTRO
Comentario: "{comentario}" ->"""
    return preguntar(prompt, proveedor=proveedor, temperatura=0.0).strip()

# Probamos con varios comentarios
for c in ["La atención fue excelente", "Tardó un mes en llegar", "Cumple su función"]:
    print(f"{clasificar_sentimiento(c):10s} <- {c}")

### 🟢 Ejercicio 1 · Cambia de proveedor

Coge cualquiera de los tres casos (chatbot, resumen o clasificación) y ejecútalo con `proveedor="openrouter"` o `proveedor="ollama"`. ¿Cambia mucho la calidad de la respuesta? Anota las diferencias que observes.

In [ ]:
# TODO: prueba un caso con otro proveedor
# Ejemplo:
# print(clasificar_sentimiento("Me ha encantado el servicio", proveedor="openrouter"))


### 🟢 Ejercicio 2 · Tu propio clasificador

Adapta la función `clasificar_sentimiento` para que clasifique correos como `URGENTE` o `NO URGENTE` (reutiliza el few-shot que diseñaste en la Sesión 5). Pruébala con 3 correos inventados.

In [ ]:
# TODO: crea la función clasificar_urgencia(correo, proveedor="claude")


**Solución de referencia:**

In [ ]:
def clasificar_urgencia(correo, proveedor="claude"):
    prompt = f"""Clasifica el correo como URGENTE o NO URGENTE.
Responde SOLO con una de esas dos opciones.

Correo: "El servidor está caído y no podemos facturar"     -> URGENTE
Correo: "¿Podríamos vernos la semana que viene?"            -> NO URGENTE
Correo: "Recordatorio: actualiza tu perfil cuando puedas"   -> NO URGENTE
Correo: "{correo}" ->"""
    return preguntar(prompt, proveedor=proveedor, temperatura=0.0).strip()

for c in ["El cliente amenaza con cancelar hoy mismo",
          "Te paso las fotos del evento del mes pasado",
          "La web de pagos da error a todos los usuarios"]:
    print(f"{clasificar_urgencia(c):12s} <- {c}")

## 10. Resumen de la sesión y cierre del Módulo 3

- Llamar a un LLM sigue siempre el mismo patrón: **cliente → lista de mensajes → leer el texto de la respuesta**
- Tienes tres vías equivalentes: **Claude** (calidad), **OpenRouter** (gratis en la nube) y **Ollama** (local y privado)
- Los parámetros clave son **temperatura** (creatividad), **max_tokens** (largo) y el **historial** (memoria)
- Programa siempre pensando en los **errores** (`try`/`except`); las APIs fallan y los límites gratuitos se agotan
- Con prompts bien diseñados resuelves **chatbot, resumen y clasificación** casi sin código

Ya sabes integrar IA generativa en tus propios programas. **Eso es exactamente lo que hace una app de IA real por dentro.**

**Para el Módulo 4:** unimos todo en un proyecto de principio a fin: de la idea a una app web publicada con Streamlit.

👉 **Trae pensado un problema pequeño y concreto** que te gustaría resolver con una app de IA. Será tu proyecto.